# Buffer + filter: can a lossier bright mode push the chaos threshold? (14/10/2026)

Mitigation point 2 of `article_chaos/report_v2.txt`. Identical parallel buffers are one bright mode with the loss of a single buffer, so they cannot help.
Here the extra mode is put **in series**: the ATS couples the memory $a$ to one buffer $b$ (no intrinsic loss), and $b$ is coupled linearly to a filter $f$ at $\omega_f=\omega_b$ which carries all the loss,
$$H \supset J(b+b^\dagger)(f+f^\dagger) = 2J\varphi_b\varphi_f ,\qquad \ddot\varphi_f = -\omega_f(\omega_f\varphi_f + 2J\varphi_b) - 2\kappa_f\dot\varphi_f .$$
Julia damping convention ($-2\kappa\dot\varphi$, field decay $\kappa$), $g_2 = E_J\sin(\varepsilon_p)\varphi_a^2\varphi_b$, $\varepsilon_d = 10.12\,g_2$ (so $|\alpha|^2 = 5.06$ for every configuration).

* The memory only sees the static response of the $b$–$f$ network, $\kappa_\mathrm{eff} = J^2/\kappa_f$ (exact at zero frequency in the RWA), so $\kappa_2 = 4g_2^2/\kappa_b^\mathrm{GKSL} = 2g_2^2/\kappa_\mathrm{eff}$ (GKSL $\kappa_b = 2\times$ code $\kappa$).
* The shape of the response depends on $r = J/\kappa_f$: $r\ll 1/2$ is a Lorentzian of width $\kappa_\mathrm{eff}$ (= a single lossy buffer), $r = 1/2$ is critical damping, $r = 1/\sqrt2$ is the maximally flat (Butterworth) two-pole filter.
* Direct-loss reference: same EOM with $J = 0$ and loss $\kappa_\mathrm{eff}$ on $b$.
* Each configuration is swept over $\varepsilon_p\in[0.2,1.5]$ and plotted against $\kappa_2$; larger $\kappa_\mathrm{eff}$ needs larger $g_2\propto\sqrt{\kappa_2\kappa_\mathrm{eff}}$, so its curve stops at a smaller $\kappa_2$ ($\sin\varepsilon_p\le1$).
* Second diagnostic: mean memory photon number $\langle|a|^2\rangle$ on the attractor reached from the origin (target 5.06), to catch a cat that shrinks or a Purcell leak through the buffer.

Outputs: `analysis/output/lyapunov_buffer_filter_14_10_2026.json`, `figures/chaos/lyapunov_buffer_filter_14_10_2026.pdf`.

In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl")
using ChaosTools, ProgressBars, JSON

In [ ]:
ϕ_quantum = 1/2
ω_a = 25.338776456203686
ω_b = 2*ω_a
ω_f = ω_b
φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
E_J = 37.12*2*pi*1.5
ω_d = ω_b                       # static pump, ω_p = 2ω_a - ω_d = 0
κ_b0 = π/10.4                   # reference buffer loss (code convention)
ϵ_d_g2_ratio = 10.12

t_max = 30/κ_b0                 # same windows for every configuration
t_w = 15/κ_b0
dt = 1/(34*sqrt(2)*ω_a)         # get_dt for MeanField
ϵ_p_list = vcat(0.2:.01:1.5)
g2_of(ϵ_p) = sin(ϵ_p)*E_J*φ_a^2*φ_b

In [ ]:
# memory a, buffer b (loss κ_b), filter f (loss κ_f), b–f coupling 2J φ_b φ_f. J = 0 gives the plain mono-buffer.
# p carries every constant (no untyped globals in the RHS: they made it ~40× slower and allocate on every call)
# u = (ϕ_a, ϕ_b, ϕ_f, ϕ̇_a, ϕ̇_b, ϕ̇_f)
function equation_1ODE_filter!(du, u, p, t)
    (; ϵ_p, ϵ_d, κ_b, J, κ_f, ω_a, ω_b, ω_f, ω_d, Z_a, Z_b, E_J, ϕ_quantum) = p
    ϕ_a, ϕ_b, ϕ_f = u[1], u[2], u[3]

    coupling = (cos(ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b * sqrt(Z_b)/ϕ_quantum) - 1) / ϕ_quantum
    sin_pump = 2*E_J*sin(ϵ_p)

    du[1] = u[4]
    du[2] = u[5]
    du[3] = u[6]
    du[4] = -ω_a*(ω_a*ϕ_a - sqrt(Z_a) * sin_pump*coupling)
    du[5] = -ω_b*(ω_b*ϕ_b + ϵ_d*cos(ω_d*t)*sqrt(2) + 2J*ϕ_f - sqrt(Z_b) * sin_pump*coupling) - 2κ_b * du[2]
    du[6] = -ω_f*(ω_f*ϕ_f + 2J*ϕ_b) - 2κ_f * du[3]
    return nothing
end

# (label, κ_eff/κ_b0, r = J/κ_f); r = 0 → direct loss on b
configs = [("direct", m, 0.0) for m in (1, 2, 3)]
append!(configs, [("filter", m, r) for m in (1, 2, 3) for r in (0.35, 0.5, 1/sqrt(2))])

function params(ϵ_p, mult, r)
    κ_eff = mult*κ_b0
    ϵ_d = ϵ_d_g2_ratio*g2_of(ϵ_p)
    consts = (; ω_a, ω_b, ω_f, ω_d, Z_a, Z_b, E_J, ϕ_quantum)
    r == 0 && return (; ϵ_p, ϵ_d, κ_b = κ_eff, J = 0.0, κ_f = 1.0, consts...)   # filter decoupled and damped
    κ_f = κ_eff/r^2                                                          # J²/κ_f = κ_eff
    return (; ϵ_p, ϵ_d, κ_b = 0.0, J = r*κ_f, κ_f, consts...)
end
[(c, "κ_f/2π (MHz) = ", round(c[3] == 0 ? 0 : c[2]*κ_b0/c[3]^2/2π*1e3, digits=0)) for c in configs]

In [ ]:
# sanity check: J = 0 must reproduce the config equation_1ODE! (κ_b0, ϵ_p = 0.7)
let ϵ_p = 0.7
    sp = SimulationParams(prepare(SystemATS(ω_a, ω_b, E_J, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0)),
                          prepare(DriveTwoModesBasics(ϵ_d_g2_ratio*g2_of(ϵ_p), ω_d, 0.0, ϵ_p, 0.0, 0.0)), MeanField(0.0, κ_b0))
    du1, du2, u = zeros(4), zeros(6), [0.3, -0.2, 0.0, 1.1, -0.7, 0.0]
    equation_1ODE!(du1, u[[1, 2, 4, 5]], sp, 0.37)
    equation_1ODE_filter!(du2, u, params(ϵ_p, 1, 0.0), 0.37)
    maximum(abs.(du1 .- du2[[1, 2, 4, 5]]))
end

In [ ]:
# λ_max from the origin and the mean memory photon number on the attractor
mean_n_a(sol) = sum(u -> (u[1]^2 + (u[4]/ω_a)^2)/2, sol.u) / length(sol.u)   # a = (ϕ + iϕ̇/ω)/√2

Lyap = zeros(length(configs), length(ϵ_p_list))
n_a = zeros(length(configs), length(ϵ_p_list))
for k in ProgressBar(1:length(configs)*length(ϵ_p_list))
    c, i = divrem(k - 1, length(ϵ_p_list)) .+ 1
    _, mult, r = configs[c]
    p = params(ϵ_p_list[i], mult, r)
    Lyap[c, i] = lyapunov(CoupledODEs(equation_1ODE_filter!, zeros(6), p), t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = t_w)
    sol = solve(ODEProblem(equation_1ODE_filter!, zeros(6), (0.0, t_max), p), Tsit5(); saveat = t_w:dt*7:t_max, reltol = 1e-6, abstol = 1e-6)
    n_a[c, i] = mean_n_a(sol)
end

In [ ]:
g2_MHz = g2_of.(ϵ_p_list)/2π*1e3
κ2_MHz(mult) = 2*g2_of.(ϵ_p_list).^2/(mult*κ_b0)/2π*1e3            # κ2 = 4g2²/κ_b^GKSL, κ_b^GKSL = 2κ_eff
config_label(c) = c[1] == "direct" ? "direct, κ_eff = $(c[2])κ_b" : "filter r = $(round(c[3], digits=2)), κ_eff = $(c[2])κ_b"

onsets = Dict{String,Any}()
for (c, cfg) in enumerate(configs)
    i = findfirst(>(1e-3), Lyap[c, :])
    onsets[config_label(cfg)] = isnothing(i) ? nothing : (g2_MHz = round(g2_MHz[i], digits=1), κ2_MHz = round(κ2_MHz(cfg[2])[i], digits=1))
    println(rpad(config_label(cfg), 34), isnothing(i) ? "no chaos in range (κ2 ≤ $(round(κ2_MHz(cfg[2])[end], digits=0)) MHz)" :
            "onset g2/2π = $(onsets[config_label(cfg)].g2_MHz) MHz, κ2/2π = $(onsets[config_label(cfg)].κ2_MHz) MHz")
end

open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/lyapunov_buffer_filter_14_10_2026.json", "w") do io
    JSON.print(io, Dict("eps_p" => ϵ_p_list, "g2_MHz" => g2_MHz, "eps_d_over_g2" => ϵ_d_g2_ratio, "kappa_b0_code" => κ_b0,
                        "configs" => [Dict("type" => c[1], "kappa_eff_over_kappa_b0" => c[2], "J_over_kappa_f" => c[3]) for c in configs],
                        "kappa2_MHz" => [κ2_MHz(c[2]) for c in configs], "lyap" => [Lyap[c, :] for c in 1:length(configs)],
                        "n_a" => [n_a[c, :] for c in 1:length(configs)], "onsets" => onsets))
end

In [ ]:
colors = Dict(1 => RGB(0, 0, 0), 2 => RGB(134/255, 187/255, 216/255), 3 => RGB(160/255, 40/255, 41/255))
styles = Dict(0.0 => :solid, 0.35 => :dot, 0.5 => :dashdot, 1/sqrt(2) => :dash)
p1 = plot(); p2 = plot()
for (c, cfg) in enumerate(configs)
    plot!(p1, κ2_MHz(cfg[2]), Lyap[c, :], color = colors[cfg[2]], ls = styles[cfg[3]], label = config_label(cfg))
    plot!(p2, κ2_MHz(cfg[2]), n_a[c, :], color = colors[cfg[2]], ls = styles[cfg[3]], label = "")
end
hline!(p1, [0], color = :gray, label = "")
hline!(p2, [ϵ_d_g2_ratio/2], color = :gray, label = "")
plot!(p1, ylabel = L"\lambda"*" (ns"*L"^{-1}"*")", legend = :topleft, legendfontsize = 6)
plot!(p2, xlabel = L"\kappa_2/2\pi"*" (MHz)", ylabel = L"\langle|a|^2\rangle", ylims = (0, 15))
plot(p1, p2, layout = (2, 1), size = (700, 700), link = :x)
#savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/chaos/lyapunov_buffer_filter_14_10_2026.pdf")